# Research Results and Statistical Analysis

## Comparative Evaluation of Machine Learning Models
## for Flow-Based DDoS Attack Detection

This notebook presents the experimental results of the research.

The analysis includes:
- Raw benchmark data ingestion.
- Warm-up filtering.
- Statistical aggregation.
- Median, p95 and p99 calculation.
- Bootstrap 95% confidence intervals.
- Error-rate monitoring.
- Scientific visualization.

Random seed: 42.

In [ ]:
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data/raw/test_run.jsonl"
PROCESSED_PATH = ROOT / "data/processed/metrics_summary.csv"
FIGURES_PATH = ROOT / "reports/figures"

FIGURES_PATH.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper")

print("Project root:", ROOT)
print("Random seed:", SEED)


## 1. Data Ingestion

The raw benchmark data is stored in JSON Lines format.

Each line represents one experimental measurement.

The dataset contains both warm-up and evaluation runs.

Warm-up runs are excluded because initial model execution
may introduce cold-start latency.

In [ ]:
records = []

with open(RAW_PATH, "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)

print("Total raw records:", len(df))
display(df.head())

In [ ]:
print("Before filtering:", len(df))

warmup_count = df["is_warmup"].astype(bool).sum()

df_eval = df[
    (df["phase"] == "evaluation") &
    (~df["is_warmup"].astype(bool))
].copy()

print("Warm-up records removed:", warmup_count)
print("Evaluation records:", len(df_eval))

assert len(df_eval) > 0, "No evaluation records found!"

display(df_eval.head())

## 2. Data Quality and Error Analysis

Failed evaluation runs are tracked as a guardrail metric.

The error rate is calculated as:

Error Rate = Failed Evaluation Runs / Total Evaluation Runs

Only successful inference measurements are used
for latency and throughput statistics.

In [ ]:
total_runs = len(df_eval)

failed_runs = df_eval["error"].astype(bool).sum()

error_rate = failed_runs / total_runs

print("Total evaluation runs:", total_runs)
print("Failed runs:", failed_runs)
print(f"Error rate: {error_rate:.4%}")

error_summary = (
    df_eval.groupby("model")
    .agg(
        total_runs=("error", "count"),
        errors=("error", "sum")
    )
)

error_summary["error_rate"] = (
    error_summary["errors"] / error_summary["total_runs"]
)

display(error_summary)

## 3. Statistical Aggregation

The following descriptive statistics are calculated:

- Median: central value of the distribution.
- p95: 95th percentile.
- p99: 99th percentile.
- 95% bootstrap confidence interval.

The confidence interval is estimated using
5000 bootstrap resamples and a fixed random seed.

In [ ]:
def bootstrap_ci(values, iterations=5000, statistic=np.median):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    if len(values) == 1:
        return float(values[0]), float(values[0])

    rng = np.random.default_rng(SEED)

    samples = rng.choice(
        values,
        size=(iterations, len(values)),
        replace=True
    )

    estimates = statistic(samples, axis=1)

    return (
        float(np.percentile(estimates, 2.5)),
        float(np.percentile(estimates, 97.5))
    )

In [ ]:
successful = df_eval[
    ~df_eval["error"].astype(bool)
].copy()

latency_results = []

for (model, batch), group in successful.groupby(
    ["model", "batch_size"]
):

    values = group["latency_ms"].dropna().to_numpy()

    ci_low, ci_high = bootstrap_ci(values)

    latency_results.append({
        "model": model,
        "batch_size": batch,
        "count": len(values),
        "median_ms": np.median(values),
        "p95_ms": np.percentile(values, 95),
        "p99_ms": np.percentile(values, 99),
        "ci95_low": ci_low,
        "ci95_high": ci_high
    })

latency_summary = pd.DataFrame(latency_results)

display(latency_summary)

In [ ]:
classification_results = []

for model, group in df_eval.groupby("model"):

    # One classification result per independent CV fold.
    folds = group.drop_duplicates(subset=["fold"])

    for metric in ["accuracy", "precision", "recall",
                   "f1_score", "fpr"]:

        values = folds[metric].dropna().to_numpy()

        low, high = bootstrap_ci(
            values,
            statistic=np.mean
        )

        classification_results.append({
            "model": model,
            "metric": metric,
            "mean": np.mean(values),
            "ci95_low": low,
            "ci95_high": high
        })

classification_summary = pd.DataFrame(
    classification_results
)

display(classification_summary)

## 4. Visualization

The first figure presents inference latency
under different batch sizes.

The shaded region represents the bootstrap
95% confidence interval of the median.

The second figure compares the distribution
of individual evaluation latency measurements
across classifiers.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

models = latency_summary["model"].unique()

markers = ["o", "s", "^", "D"]
linestyles = ["-", "--", "-.", ":"]

for i, model in enumerate(models):

    part = latency_summary[
        latency_summary["model"] == model
    ].sort_values("batch_size")

    x = part["batch_size"].to_numpy()
    y = part["median_ms"].to_numpy()
    low = part["ci95_low"].to_numpy()
    high = part["ci95_high"].to_numpy()

    ax.plot(
        x, y,
        marker=markers[i % len(markers)],
        linestyle=linestyles[i % len(linestyles)],
        label=model
    )

    ax.fill_between(x, low, high, alpha=0.15)

ax.set_title("Inference Latency vs Batch Size")
ax.set_xlabel("Batch size (samples)")
ax.set_ylabel("Inference latency (ms)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "fig1_latency_vs_load.pdf",
    bbox_inches="tight"
)

plt.savefig(
    FIGURES_PATH / "fig1_latency_vs_load.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=successful,
    x="model",
    y="latency_ms",
    hue="model",
    palette="colorblind",
    legend=False,
    ax=ax
)

ax.set_title("Distribution of Inference Latency")
ax.set_xlabel("Machine Learning Model")
ax.set_ylabel("Inference latency (ms)")
ax.tick_params(axis="x", rotation=20)

plt.tight_layout()

plt.savefig(
    FIGURES_PATH / "fig2_distribution_boxplot.pdf",
    bbox_inches="tight"
)

plt.savefig(
    FIGURES_PATH / "fig2_distribution_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
summary = latency_summary.merge(
    error_summary.reset_index(),
    on="model",
    how="left"
)

summary.to_csv(PROCESSED_PATH, index=False)

print("Final summary saved:", PROCESSED_PATH)

display(summary)

## 5. Results and Discussion

The experimental results provide a comparison
of the computational performance of the selected
machine learning classifiers.

Latency measurements show how inference time
changes with batch size.

The boxplot provides information about the
distribution and variability of inference latency.

The classification metrics provide a separate
evaluation of detection quality.

The error rate is monitored as a guardrail metric.

The results should be interpreted in the context
of the dataset size and experimental setup.

A small sample dataset limits the generalizability
of the findings to real-world network traffic.